In [0]:
# Simulates what a Lakeflow Connect SQL Server pipeline would land in the restricted zone (step 0).
# Runs as the pseudonymization SP. Synthetic data only. Idempotent per batch_id.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
dbutils.widgets.text("batch_id", "1")
dbutils.widgets.dropdown("scenario", "baseline", ["baseline", "drift_and_bad_data"])
catalog = dbutils.widgets.get("catalog")
batch_id = int(dbutils.widgets.get("batch_id"))
scenario = dbutils.widgets.get("scenario")
RAW = f"{catalog}.pii_raw"
CUSTOMERS, TRANSACTIONS = f"{RAW}.lfc_customers", f"{RAW}.lfc_transactions"

In [0]:
import random, datetime
from decimal import Decimal

FIRST = ["Alice", "Bob", "Carol", "David", "Eva", "Frank", "Grace", "Hugo", "Ines", "Joao", "Karin", "Liam", "Maria", "Nuno", "Olga", "Pedro", "Rita", "Sven", "Tara", "Uma"]
LAST = ["Johnson", "Smith", "White", "Lee", "Martinez", "Silva", "Costa", "Muller", "Rossi", "Dubois", "Novak", "Jensen", "Kowalski", "Santos", "Brown"]
DOMAINS = ["gmail.com", "outlook.com", "yahoo.com", "proton.me", "example.org"]
CITIES = ["Lisbon", "Porto", "Madrid", "Berlin", "Paris", "Amsterdam", "Dublin", "New York", "Chicago", "Austin"]
SEGMENTS = ["retail", "premium", "business", "student"]
MERCHANTS = ["Amazon", "Uber", "Starbucks", "Lidl", "Zara", "Shell", "Netflix", "IKEA", "Apple Store", "Booking.com"]
PHONE_FORMATS = ["({a}) {b}-{c}", "{a}.{b}.{c}", "+1 {a} {b} {c}", "1-{a}-{b}-{c}"]

def customer(i: int) -> dict:
    """Deterministic per customer_id, so re-sent rows (CDC updates) carry the same PII."""
    r = random.Random(i)
    first, last = r.choice(FIRST), r.choice(LAST)
    a, b, c = r.randint(200, 989), r.randint(100, 999), r.randint(1000, 9999)
    return {
        "customer_id": i,
        "full_name": f"{first} {last}",
        "email": f"{first.lower()}.{last.lower()}{i}@{r.choice(DOMAINS)}",
        "phone": r.choice(PHONE_FORMATS).format(a=a, b=b, c=c),
        "ssn": f"{r.randint(100, 899):03d}-{r.randint(10, 99):02d}-{r.randint(1000, 9999):04d}",
        "dob": datetime.date(1950, 1, 1) + datetime.timedelta(days=r.randint(0, 18000)),
        "street_address": f"{r.randint(1, 999)} {r.choice(['Main St', 'Rua Augusta', 'Gran Via', 'Unter den Linden'])}",
        "city": r.choice(CITIES),
        "segment": r.choice(SEGMENTS),
        "signup_date": datetime.date(2020, 1, 1) + datetime.timedelta(days=r.randint(0, 2000)),
    }

def inject_bad(row: dict, i: int, rate_mod: int) -> dict:
    """About 5% invalid rows in batch 1, more in batch 2, so quarantine has something to show."""
    if i % rate_mod == 0:
        row["email"] = row["email"].replace("@", ".at.")          # invalid email format
    if i % 33 == 0:
        row["ssn"] = "12345"                                      # wrong SSN length
    if i % 47 == 0:
        row["phone"] = "n/a"                                      # unparseable phone
    return row

In [0]:
rng = random.Random(1000 + batch_id)
customers, transactions = [], []

if scenario == "baseline":
    ids = list(range(1, 201))
    customers = [inject_bad(customer(i), i, 20) for i in ids]
    customers.append({**customer(999), "customer_id": None})            # missing primary key
    txn_customer_ids = ids
else:
    # drift_and_bad_data: new customers, re-sent existing customers (CDC updates with messy formatting),
    # an undeclared column (schema drift), more invalid emails, orphan and leaking transactions.
    ids = list(range(201, 241))
    customers = [inject_bad(customer(i), i, 8) for i in ids]
    for i in range(1, 11):
        row = customer(i)
        row["email"] = "  " + row["email"].upper() + " "                  # same person, messy format -> same token after normalization
        row["segment"] = "premium"
        customers.append(row)
    for row in customers[:5]:
        row["notes"] = f"VIP - call {row['full_name']} at {row['phone']}"  # undeclared column carrying PII
    txn_customer_ids = list(range(1, 241))

for n in range(600 if scenario == "baseline" else 150):
    amount = Decimal(rng.randint(100, 50000)) / 100
    transactions.append({
        "txn_id": f"T{batch_id}{n:06d}",
        "customer_id": rng.choice(txn_customer_ids),
        "amount": -amount if n % 50 == 0 else amount,                     # some negative amounts -> quarantine
        "currency": "XXX" if n % 77 == 0 else rng.choice(["EUR", "USD", "GBP"]),
        "merchant": rng.choice(MERCHANTS),
        "card_number": "4" + "".join(str(rng.randint(0, 9)) for _ in range(15)),
        "txn_ts": datetime.datetime(2026, 11, 1) + datetime.timedelta(minutes=rng.randint(0, 30000)),
    })
if scenario != "baseline":
    transactions[0]["customer_id"] = 9999                                # orphan (referential integrity)
    transactions[1]["merchant"] = "REFUND TO jane.doe@example.com"        # planted cleartext leak in a 'keep' column
print(f"batch {batch_id}: {len(customers)} customers, {len(transactions)} transactions")

In [0]:
from pyspark.sql import functions as F, types as T

cust_fields = [
    T.StructField("customer_id", T.LongType()), T.StructField("full_name", T.StringType()),
    T.StructField("email", T.StringType()), T.StructField("phone", T.StringType()),
    T.StructField("ssn", T.StringType()), T.StructField("dob", T.DateType()),
    T.StructField("street_address", T.StringType()), T.StructField("city", T.StringType()),
    T.StructField("segment", T.StringType()), T.StructField("signup_date", T.DateType()),
]
if scenario != "baseline":
    cust_fields.append(T.StructField("notes", T.StringType()))
txn_schema = T.StructType([
    T.StructField("txn_id", T.StringType()), T.StructField("customer_id", T.LongType()),
    T.StructField("amount", T.DecimalType(10, 2)), T.StructField("currency", T.StringType()),
    T.StructField("merchant", T.StringType()), T.StructField("card_number", T.StringType()),
    T.StructField("txn_ts", T.TimestampType()),
])

def land(rows, schema, table, source):
    if spark.catalog.tableExists(table) and spark.table(table).where(F.col("_batch_id") == batch_id).limit(1).count() > 0:
        print(f"{table}: batch {batch_id} already landed, skipping")
        return
    df = (spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)
          .withColumn("_source_system", F.lit(source))
          .withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_batch_id", F.lit(batch_id)))
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(table)
    # Raw zone keeps cleartext only briefly: short file retention so VACUUM really removes old cleartext.
    spark.sql(f"""ALTER TABLE {table} SET TBLPROPERTIES (
        'delta.deletedFileRetentionDuration' = 'interval 7 days', 'delta.logRetentionDuration' = 'interval 7 days')""")
    print(f"{table}: landed {len(rows)} rows for batch {batch_id}")

land(customers, T.StructType(cust_fields), CUSTOMERS, "sqlserver://crm/dbo.customers")
land(transactions, txn_schema, TRANSACTIONS, "sqlserver://crm/dbo.transactions")
display(spark.sql(f"SELECT _batch_id, count(*) AS n FROM {CUSTOMERS} GROUP BY ALL ORDER BY 1"))